In [6]:
import pandas as pd
import io

# Paste each lot as a text block, separated by ---
raw_lots = """
artist	Damien Hirst
title	Beautiful, Absolutely Fresh, Truly Wonderful, Inconceivably Drear, Still Distant, Bright And Glorious Gyration Painting
medium	Giclée print in colours on polycotton artist's stretched canvas
height_cm	115
width_cm	9
estimate_low	2500
estimate_high	3500
realized_price	
sold	0
auction_house	Forum Auctions
sale_date	2026-05-13
---
artist	Damien Hirst
title	To Belong, from Butterfly Etchings
medium	Etching with aquatint
height_cm	41
width_cm	44.5
estimate_low	3000
estimate_high	5000
realized_price	
sold	0
auction_house	Forum Auctions
sale_date	2026-05-13
---

"""

# Parse each lot block into a dictionary
records = []
for lot in raw_lots.strip().split('---'):
    lot = lot.strip()
    if not lot:
        continue
    record = {}
    for line in lot.split('\n'):
        line = line.strip()
        if '\t' in line:
            key, value = line.split('\t', 1)
            record[key.strip()] = value.strip()
    if record:
        records.append(record)

df = pd.DataFrame(records)

# Clean up numeric columns
for col in ['height_cm', 'width_cm', 'estimate_low', 'estimate_high', 'sold']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Total records: {len(df)}")
df

Total records: 2


,artist,title,medium,height_cm,width_cm,estimate_low,estimate_high,sold,auction_house,sale_date
0,Damien Hirst,"Beautiful, Absolutely Fresh, Truly Wonderful, ...",Giclée print in colours on polycotton artist's...,115,9.0,2500,3500,0,Forum Auctions,2026-05-13
1,Damien Hirst,"To Belong, from Butterfly Etchings",Etching with aquatint,41,44.5,3000,5000,0,Forum Auctions,2026-05-13


In [ ]:
from google.colab import files
df.to_csv('artsy_auction_data.csv', index=False)
files.download('artsy_auction_data.csv')
print("Downloaded!")

In [23]:
import pdfplumber
import re
import pandas as pd

pdf_path = r'C:\Users\Nathan\art1.pdf'

full_text = ''
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            full_text += ' '.join(text.split()) + ' '

splits = [m.start() for m in re.finditer(r'artistNames', full_text)]
print(f"Found {len(splits)} artwork blocks")

records = []

for i, start in enumerate(splits):
    end = splits[i+1] if i+1 < len(splits) else len(full_text)
    chunk = full_text[start:min(end, start+5000)]

    def get_val(pattern, text):
        m = re.search(pattern, text)
        return m.group(1).strip() if m else ''

    artist   = get_val(r'artistNames\\":\\"([^"\\\\]+)', chunk)
    title    = get_val(r'\\"title\\":\\"([^"\\\\]+)\\".*?date', chunk)
    medium   = get_val(r'\\"medium\\":\\"([^"\\\\]+)\\"', chunk)
    height   = get_val(r'\\"heightCm\\":([\d.]+)', chunk)
    width    = get_val(r'\\"widthCm\\":([\d.]+)', chunk)
    is_sold  = get_val(r'\\"isSold\\":(true|false)', chunk)
    partner  = get_val(r'\\"partner\\".*?\\"name\\":\\"([^"\\\\]+)\\"', chunk)
    end_date = get_val(r'\\"endAt\\":\\"([^"\\\\]+)\\"', chunk)
    currency = get_val(r'\\"currency\\":\\"([A-Z]{3})\\"', chunk)
    est_raw  = get_val(r'\\"estimate\\":\\"([^"\\\\]+)\\"', chunk)

    if i < 3:
        print(f"\n--- Block {i+1} ---")
        print(f"artist: {artist}")
        print(f"title: {title}")
        print(f"estimate: {est_raw}")
        print(f"sold: {is_sold}")

    estimate_low = None
    estimate_high = None
    if est_raw:
        nums = re.findall(r'[\d,]+', est_raw)
        if len(nums) >= 2:
            estimate_low  = int(nums[0].replace(',',''))
            estimate_high = int(nums[1].replace(',',''))
        elif len(nums) == 1:
            estimate_low = int(nums[0].replace(',',''))

    if artist:
        records.append({
            'artist':         artist,
            'title':          title,
            'medium':         medium,
            'height_cm':      float(height) if height else None,
            'width_cm':       float(width) if width else None,
            'estimate_low':   estimate_low,
            'estimate_high':  estimate_high,
            'realized_price': None,
            'sold':           1 if is_sold == 'true' else 0,
            'auction_house':  partner,
            'sale_date':      end_date[:10] if end_date else '',
            'currency':       currency
        })

df = pd.DataFrame(records)
if len(df) > 0:
    df = df.drop_duplicates(subset=['artist','title'])
    print(f"\nTotal unique records: {len(df)}")
    print(df.to_string())
else:
    print("No records found - check debug output above")

Found 32 artwork blocks

--- Block 1 ---
artist: Banksy
title: 
estimate: 
sold: false

--- Block 2 ---
artist: Banksy
title: 
estimate: 
sold: false

--- Block 3 ---
artist: Banksy
title: 
estimate: 
sold: false

Total unique records: 9
              artist                                                                 title                                                medium  height_cm  width_cm estimate_low estimate_high realized_price  sold   auction_house sale_date currency
0             Banksy                                                                                                      Screenprint in colours       69.7       NaN         None          None           None     0                                   
4            Invader                                                                                                               Ceramic tiles       18.0      24.0         None          None           None     0  Forum Auctions                   
6      Grayson 